<a href="https://colab.research.google.com/github/mhesham0011-rgb/VT_LOG_IP_CHECK/blob/main/SOC_Assistant_API_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# SOC Assistant API Platform
# A comprehensive, ready-to-deploy API for common Security Operations Center tasks.
# This single file contains the entire application logic.

import random
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional

from fastapi import FastAPI, Path, HTTPException
from pydantic import BaseModel, Field

# --- Pydantic Models: Define the structure of API responses for data validation ---

class IPInfoResponse(BaseModel):
    """Response model for IP address intelligence."""
    ip_address: str = Field(..., description="The IP address that was queried.")
    is_malicious: bool = Field(..., description="Flag indicating if the IP is considered malicious.")
    score: int = Field(..., description="A reputation score from 0 (good) to 100 (bad).")
    geolocation: Dict[str, Any] = Field(..., description="Geographical information for the IP.")
    asn: Dict[str, Any] = Field(..., description="Autonomous System Number (ASN) details.")
    blocklist_hits: int = Field(..., description="Number of blocklists the IP appears on.")
    related_domains: List[str] = Field(..., description="Domains recently associated with this IP.")
    last_seen: datetime = Field(..., description="Timestamp of the last malicious activity observed.")

class DomainInfoResponse(BaseModel):
    """Response model for domain name intelligence."""
    domain: str = Field(..., description="The domain name that was queried.")
    is_malicious: bool = Field(..., description="Flag indicating if the domain is considered malicious.")
    score: int = Field(..., description="A reputation score from 0 (good) to 100 (bad).")
    registrar: str = Field(..., description="The registrar of the domain.")
    creation_date: datetime = Field(..., description="The date the domain was created.")
    tags: List[str] = Field(..., description="Tags associated with the domain (e.g., phishing, malware).")
    resolutions: List[str] = Field(..., description="Recent IP addresses the domain has resolved to.")

class HashInfoResponse(BaseModel):
    """Response model for file hash analysis."""
    file_hash: str = Field(..., description="The file hash (MD5, SHA1, SHA256) that was queried.")
    is_malicious: bool = Field(..., description="Flag indicating if the hash corresponds to a known malicious file.")
    detection_rate: str = Field(..., description="The detection rate from simulated AV engines (e.g., '57/70').")
    first_seen: datetime = Field(..., description="Timestamp when this file was first seen in the wild.")
    signatures: List[str] = Field(..., description="Common names or signatures for the malware.")
    file_type: str = Field(..., description="The detected type of the file (e.g., PE, PDF, ELF).")

class CVEInfoResponse(BaseModel):
    """Response model for CVE details."""
    cve_id: str = Field(..., description="The CVE identifier that was queried.")
    summary: str = Field(..., description="A brief summary of the vulnerability.")
    cvss_score: float = Field(..., description="The CVSS v3.1 base score for the vulnerability.")
    severity: str = Field(..., description="The severity level (e.g., CRITICAL, HIGH, MEDIUM, LOW).")
    published_date: datetime = Field(..., description="The date the CVE was published.")
    references: List[str] = Field(..., description="A list of reference URLs for more details.")


# --- Mock Threat Intelligence Services ---
# In a real-world application, these functions would make calls to external APIs
# like VirusTotal, AbuseIPDB, Shodan, etc. Here, we simulate that behavior.

def get_mock_ip_reputation(ip_address: str) -> Dict[str, Any]:
    """Simulates fetching IP reputation from a threat intelligence provider."""
    is_malicious = random.choice([True, False, False]) # Skew towards not malicious
    score = random.randint(70, 100) if is_malicious else random.randint(0, 30)
    return {
        "ip_address": ip_address,
        "is_malicious": is_malicious,
        "score": score,
        "geolocation": {"country": "Utopia", "city": "Cyberville", "lat": 12.34, "lon": 56.78},
        "asn": {"number": f"AS{random.randint(1000, 65000)}", "organization": "Mock ISP Inc."},
        "blocklist_hits": random.randint(5, 20) if is_malicious else 0,
        "related_domains": [f"bad-site-{i}.com" for i in range(random.randint(1, 3))] if is_malicious else ["good-site.net"],
        "last_seen": datetime.now() - timedelta(days=random.randint(1, 100))
    }

def get_mock_domain_reputation(domain: str) -> Dict[str, Any]:
    """Simulates fetching domain reputation."""
    is_malicious = random.choice([True, False, False, False])
    tags = ["phishing", "malware_c2"] if is_malicious else ["benign", "cdn"]
    return {
        "domain": domain,
        "is_malicious": is_malicious,
        "score": random.randint(80, 100) if is_malicious else random.randint(0, 10),
        "registrar": "Mock Registrar LLC",
        "creation_date": datetime.now() - timedelta(days=random.randint(30, 3650)),
        "tags": tags,
        "resolutions": [f"192.168.1.{random.randint(1, 254)}"] if is_malicious else [f"10.0.0.{random.randint(1,254)}"]
    }

def get_mock_hash_reputation(file_hash: str) -> Dict[str, Any]:
    """Simulates fetching file hash reputation."""
    is_malicious = len(file_hash) > 40 and random.choice([True, True, False]) # Longer hashes more likely malicious
    return {
        "file_hash": file_hash,
        "is_malicious": is_malicious,
        "detection_rate": f"{random.randint(40, 70)}/70" if is_malicious else "0/70",
        "first_seen": datetime.now() - timedelta(days=random.randint(10, 500)),
        "signatures": ["Trojan.Generic.KD", "WannaCry"] if is_malicious else ["Clean"],
        "file_type": "Windows PE"
    }

def get_mock_cve_details(cve_id: str) -> Dict[str, Any]:
    """Simulates fetching CVE details from a vulnerability database."""
    score = round(random.uniform(4.0, 10.0), 1)
    if score >= 9.0:
        severity = "CRITICAL"
    elif score >= 7.0:
        severity = "HIGH"
    elif score >= 4.0:
        severity = "MEDIUM"
    else:
        severity = "LOW"

    return {
        "cve_id": cve_id,
        "summary": f"This is a mock summary for {cve_id}. It describes a simulated vulnerability allowing for remote code execution via a buffer overflow in a popular web server.",
        "cvss_score": score,
        "severity": severity,
        "published_date": datetime.now() - timedelta(days=random.randint(30, 2000)),
        "references": [
            f"https://nvd.nist.gov/vuln/detail/{cve_id}",
            f"https://attackerkb.com/vulnerabilities/{cve_id}",
        ]
    }


# --- FastAPI Application Setup ---

app = FastAPI(
    title="SOC Assistant API",
    description="An API platform providing essential tools for Security Operations Center (SOC) analysts. This includes threat intelligence lookups for IPs, domains, file hashes, and CVEs.",
    version="1.0.0",
    contact={
        "name": "SOC Platform Admin",
        "url": "https://github.com/your-repo",
        "email": "admin@example.com",
    },
)

# --- API Endpoints ---

@app.get("/", tags=["General"])
async def root():
    """Root endpoint to welcome users and confirm the API is running."""
    return {"message": "Welcome to the SOC Assistant API. See /docs for available endpoints."}

@app.get("/ip/{ip_address}", response_model=IPInfoResponse, tags=["Threat Intelligence"])
async def get_ip_info(
    ip_address: str = Path(..., description="The IPv4 or IPv6 address to query.", regex=r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$|^([0-9a-fA-F]{1,4}:){7,7}[0-9a-fA-F]{1,4}|([0-9a-fA-F]{1,4}:){1,7}:|([0-9a-fA-F]{1,4}:){1,6}:[0-9a-fA-F]{1,4}|([0-9a-fA-F]{1,4}:){1,5}(:[0-9a-fA-F]{1,4}){1,2}|([0-9a-fA-F]{1,4}:){1,4}(:[0-9a-fA-F]{1,4}){1,3}|([0-9a-fA-F]{1,4}:){1,3}(:[0-9a-fA-F]{1,4}){1,4}|([0-9a-fA-F]{1,4}:){1,2}(:[0-9a-fA-F]{1,4}){1,5}|[0-9a-fA-F]{1,4}:((:[0-9a-fA-F]{1,4}){1,6})|:((:[0-9a-fA-F]{1,4}){1,7}|:)|fe80:(:[0-9a-fA-F]{0,4}){0,4}%[0-9a-zA-Z]{1,}|::(ffff(:0{1,4}){0,1}:){0,1}((25[0-5]|(2[0-4]|1{0,1}[0-9]){0,1}[0-9])\.){3,3}(25[0-5]|(2[0-4]|1{0,1}[0-9]){0,1}[0-9])|([0-9a-fA-F]{1,4}:){1,4}:((25[0-5]|(2[0-4]|1{0,1}[0-9]){0,1}[0-9])\.){3,3}(25[0-5]|(2[0-4]|1{0,1}[0-9]){0,1}[0-9])$")
):
    """
    Retrieves threat intelligence information for a given IP address.

    This endpoint simulates a lookup against IP reputation databases and blocklists.
    """
    result = get_mock_ip_reputation(ip_address)
    return result

@app.get("/domain/{domain_name}", response_model=DomainInfoResponse, tags=["Threat Intelligence"])
async def get_domain_info(
    domain_name: str = Path(..., description="The domain name to query.")
):
    """
    Retrieves threat intelligence information for a given domain name.

    This simulates checking domain reputation, WHOIS data, and known associations.
    """
    result = get_mock_domain_reputation(domain_name)
    return result

@app.get("/hash/{file_hash}", response_model=HashInfoResponse, tags=["Threat Intelligence"])
async def get_hash_info(
    file_hash: str = Path(..., description="The MD5, SHA1, or SHA256 hash to query.")
):
    """

    Retrieves threat intelligence for a file hash from antivirus engines.

    Simulates a lookup against a service like VirusTotal to see if the hash
    is associated with known malware.
    """
    if len(file_hash) not in [32, 40, 64]:
        raise HTTPException(status_code=400, detail="Invalid hash length. Must be 32 (MD5), 40 (SHA1), or 64 (SHA256).")
    result = get_mock_hash_reputation(file_hash)
    return result

@app.get("/cve/{cve_id}", response_model=CVEInfoResponse, tags=["Vulnerability Lookup"])
async def get_cve_info(
    cve_id: str = Path(..., description="The CVE identifier (e.g., CVE-2021-44228).", regex=r"^CVE-\d{4}-\d{4,}$")
):
    """
    Retrieves details for a specific Common Vulnerabilities and Exposures (CVE) ID.

    Provides a summary, CVSS score, and references for the vulnerability.
    """
    result = get_mock_cve_details(cve_id)
    return result

# --- Uvicorn Runner ---
# This block allows running the application directly with `python soc_api_platform.py`
# for local development and debugging. It's not used when running with Docker.
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)